# bedrock — camelCase usage, and an id the price table has never heard of

Two things about Bedrock bite quietly: usage comes back camelCase one level down, and every model id is a marketplace id, so a USD budget projects `$0` and **silently never binds**.

> **Offline.** No API key, no network — the provider is a fake with the real client's *shape*, or a
> committed cassette. This notebook runs in CI on Python 3.11 and 3.13 via `nbmake`, so if a cell
> below stops working the build goes red.
>
> Beside it, [`main.py`](main.py) is the same story as a script. The last cell here asserts what
> that script asserts.

In [ ]:
# The notebook sits beside the recipe, so its own module is importable. Everything below reuses the
# recipe's fixtures rather than re-inventing them — a notebook that built its own fake could drift
# away from what `main.py` proves and nobody would notice.
import pathlib
import sys

sys.path.insert(0, str(pathlib.Path.cwd()))

## The five steps

Every recipe in `providers/` walks the same five, in the same order:

| # | Step | Here |
|---|---|---|
| 1 | **connect** | `boto3.client("bedrock-runtime")` — the `converse` shape |
| 2 | **instrument** | one wrap — detection is structural, not name-based |
| 3 | **govern** | a `tokenguard` budget **and** a `guardrails` gate |
| 4 | **record** | `cassette` — the same call replayed offline, 0 provider calls |
| 5 | **prove** | `acttrace` `verify()` and a cost that came from `prices` |

⚠️ **Not every Bedrock id is unpriced.** The lookup strips the region prefix, the vendor prefix and `-v1:0`, so a *current* Bedrock Claude id prices itself with no registration while Nova / Llama / Mistral and *retired* Claude ids do not.

## 1–2 · Connect and instrument

In [ ]:
import main as recipe
from cendor.core import bus, instrument, prices
from cendor.core.types import LLMCall

seen = []
bus.subscribe(lambda e: seen.append(e) if isinstance(e, LLMCall) else None)
client = instrument(recipe.fake_bedrock_runtime())
recipe.ask(client)
call = seen[-1]
print(f"provider: {call.provider}   (detected from the boto-shaped .converse method)")
print(f"usage   : {call.usage.input_tokens} in + {call.usage.output_tokens} out")
print(f"cost    : {call.cost}   <- no price row for this id")

## 3 · The cap that binds with no rate at all

A **token** budget counts what the provider reported. Deliberately set below one call's settled usage, because that is the honest edge: `block` is *pre-flight*.

In [ ]:
from cendor.tokenguard import BudgetExceeded, budget, reset

reset()
try:
    with budget(tokens=1_000, on_exceed="block"):
        recipe.ask(client)
        recipe.ask(client)
except BudgetExceeded as e:
    print(e)

## 3b · And the one that needs a rate you supply

In [ ]:
prices.register_model_price(recipe.MODEL_ID, input=0.06, output=0.24)  # USD / 1M tokens
reset()
with budget(usd=1.00, on_exceed="block"):
    recipe.ask(client)
print(f"priced: ${seen[-1].cost.amount}   (same id, same call, now costed)")

## 5 · Prove it

In [ ]:
assert call.usage.input_tokens == 1_100, "camelCase inputTokens was not normalized"
assert call.cost is None, "this id is expected to be UNPRICED before registration"
assert seen[-1].cost.amount > 0, "registration did not make the id priceable"
print("OK")